## Level 1

In [1]:
map_dict = {
    '1': '!',
    '2': '@',
    '3': "#",
    '4': "$",
    '5': "%",
    '6': "^",
    '7': "&",
    '8': "*",
    '9': "(",
    '0': ")"
}

text = "04your1235name6789"
print(f"Input: \t\t {text}")
result = ''.join(map_dict.get(ch, ch) for ch in text)
print(f"Result: \t {result}")

# reverse mapping
rev_map = {v: k for k, v in map_dict.items()}
result = ''.join(rev_map.get(ch, ch) for ch in text)
print(f"Recover-Result:\t{result}")

Input: 		 04your1235name6789
Result: 	 )$your!@#%name^&*(
Recover-Result:	04your1235name6789


## Level 2

One-One mapping

In [2]:
import random

def encrypt_shuffle(s, key):
    idx = list(range(len(s)))
    random.seed(key)
    random.shuffle(idx)
    return ''.join(s[i] for i in idx), idx

def decrypt_shuffle(cipher, idx):
    res = [''] * len(cipher)
    for i, j in enumerate(idx):
        res[j] = cipher[i]
    return ''.join(res)

key = 69
res, idxs = encrypt_shuffle(text, key)
print(f"Encrypt: {res}")
print(f"indexes: {idxs}")

res = decrypt_shuffle(res, idxs)
print(f"Decrypt: {res}")

Encrypt: 06ayn8ue213m759ro4
indexes: [0, 14, 11, 2, 10, 16, 4, 13, 7, 6, 8, 12, 15, 9, 17, 5, 3, 1]
Decrypt: 04your1235name6789


## Level 3.

In [3]:
import string

def encrypt_substitution(s, key):
    random.seed(key)
    alphabet = string.printable
    shuffled = list(alphabet)
    random.shuffle(shuffled)
    table = dict(zip(alphabet, shuffled))
    return ''.join(table[c] for c in s)


def decrypt_substitution(cipher, key):
    random.seed(key)
    alphabet = string.printable
    shuffled = list(alphabet)
    random.shuffle(shuffled)
    table = dict(zip(shuffled, alphabet))
    return ''.join(table[c] for c in cipher)

res = encrypt_substitution(text, key)
print(f"Encrypt: {res}")

res = decrypt_substitution(res, key)
print(f"Decrypt: {res}")

Encrypt: b.7s!kB03e_L(@Gwt<
Decrypt: 04your1235name6789


## Level 4

In [5]:
def _prepare_vigenere_key(key) -> str:
    """
    Prepares the Vigenere key by ensuring it's a string.
    Handles int, float, str, and bytes types.
    """
    if isinstance(key, (int, float)):
        key_str = str(key)
    elif isinstance(key, str):
        key_str = key
    elif isinstance(key, bytes):
        # Assuming bytes keys are intended to be decoded text
        key_str = key.decode('utf-8')
    else:
        raise TypeError(f"Unsupported key type: {type(key)}. Key must be str, int, float, or bytes.")

    if not key_str:
        raise ValueError("Vigenere key cannot be an empty string after conversion.")
    return key_str

def encrypt_vigenere(s: str, key) -> str:
    key_str = _prepare_vigenere_key(key)
    return ''.join(
        chr((ord(c) + ord(key_str[i % len(key_str)])) % 256)
        for i, c in enumerate(s)
    )

def decrypt_vigenere(c: str, key) -> str:
    key_str = _prepare_vigenere_key(key)
    return ''.join(
        chr((ord(ch) - ord(key_str[i % len(key_str)])) % 256)
        for i, ch in enumerate(c)
    )

res = encrypt_vigenere(text, key)
print(f"Encrypt: {res}")

res = decrypt_vigenere(res, key)
print(f"Decrypt: {res}")

Encrypt: fm¯¨««gkin¤£lpnr
Decrypt: 04your1235name6789


## Level 5

In [6]:
import random

def encrypt_stream_weak(data: bytes, seed: int) -> bytes:
    rnd = random.Random(seed)
    return bytes(b ^ rnd.getrandbits(8) for b in data)

def decrypt_stream_weak(cipher: bytes, seed: int) -> bytes:
    return encrypt_stream_weak(cipher, seed)

text_bytes = text.encode('utf-8')

res_bytes = encrypt_stream_weak(text_bytes, key)
print(f"Encrypt: {res_bytes}")

res_decrypted_bytes = decrypt_stream_weak(res_bytes, key)
# Convert the decrypted bytes back to a string for printing if desired
print(f"Decrypt: {res_decrypted_bytes.decode('utf-8')}")

Encrypt: b'\x9f=a\xa2_c\xabj`\xde\x84\xbe\xa4\xae\xeb\xbbRO'
Decrypt: 04your1235name6789


## Level 6

In [7]:
def otp_encrypt(data: bytes, key: bytes) -> bytes:
    assert len(data) == len(key), f"Data length ({len(data)}) must match key length ({len(key)})"
    return bytes(d ^ k for d, k in zip(data, key))

def otp_decrypt(cipher: bytes, key: bytes) -> bytes:
    return otp_encrypt(cipher, key)

plaintext_bytes = text.encode('utf-8')

encrypted_otp_result = otp_encrypt(plaintext_bytes, res_bytes)
print(f"Encrypt: {encrypted_otp_result}")

decrypted_otp_result = otp_decrypt(encrypted_otp_result, res_bytes)
print(f"Decrypt: {decrypted_otp_result.decode('utf-8')}")

Encrypt: b'\xaf\t\x18\xcd*\x11\x9aXS\xeb\xea\xdf\xc9\xcb\xdd\x8cjv'
Decrypt: 04your1235name6789


## Level 7

In [9]:
from hashlib import sha256

def xor_cipher(data: bytes, key: bytes) -> bytes:
    return bytes(d ^ key[i % len(key)] for i, d in enumerate(data))

def derive_key_from_password(password: str, length: int) -> bytes:
    # password argument expects a string, then it encodes it to bytes
    digest = sha256(password.encode()).digest()
    return (digest * (length // len(digest) + 1))[:length]


def encrypt_password_based(data: bytes, password: str) -> bytes:
    key = derive_key_from_password(password, len(data))
    return xor_cipher(data, key)

def decrypt_password_based(cipher: bytes, password: str) -> bytes:
    # This function expects 'password' to be a string
    return encrypt_password_based(cipher, password)

encrypted_pb_result = encrypt_password_based(plaintext_bytes, text)
print(f"Encrypt: {encrypted_pb_result}")

decrypted_pb_result = decrypt_password_based(encrypted_pb_result, text)
print(f"Decrypt: {decrypted_pb_result.decode('utf-8')}")

Encrypt: b'\xebE\xf6\x87\xbe\x83L9U<\xd7\x90\xbdH\x8do\xc5\xa4'
Decrypt: 04your1235name6789


## Level 8

In [11]:
import os
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

def encrypt_aes_gcm(data: bytes, key: bytes) -> bytes:
    aes = AESGCM(key)
    nonce = os.urandom(12)
    return nonce + aes.encrypt(nonce, data, None)

def decrypt_aes_gcm(cipher: bytes, key: bytes) -> bytes:
    nonce, ct = cipher[:12], cipher[12:]
    aes = AESGCM(key)
    return aes.decrypt(nonce, ct, None)

# Generate a new, appropriately sized key for AESGCM (e.g., 16 bytes for 128-bit AES)
aes_gcm_key = os.urandom(16) 

encrypted_aes_gcm_result = encrypt_aes_gcm(plaintext_bytes, aes_gcm_key)
print(f"Encrypt: {encrypted_aes_gcm_result}")

decrypted_aes_gcm_result = decrypt_aes_gcm(encrypted_aes_gcm_result, aes_gcm_key)
print(f"Decrypt: {decrypted_aes_gcm_result.decode('utf-8')}")

Encrypt: b'\xc3M\xf7&\xf5\x1eWQ;\x0f\xfc.gr\xd8$\xb6\xfcC\x9d$|\xb2V\xffB\nt\xbe\xfds\x83x\xa7p\xae\xd3\x16\x04YWOk\xa8\xb5j'
Decrypt: 04your1235name6789
